# TGUAI — Анализ Output Manipulation

Сравнение трёх стратегий промптинга для сегментации книжных отзывов:
- **baseline** — свободный промпт
- **structured** — JSON-схема + whitelist меток
- **few_shot** — 2 примера + строгая схема

In [ ]:
import sys
import json
import random
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

# Добавляем путь к модулю
sys.path.insert(0, '.')
from segmenter_module import run_experiment, summarize_results, VALID_LABELS

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.style.use('seaborn-v0_8-whitegrid')
print('Модули загружены успешно')

## 1. Запуск эксперимента

Убедитесь, что в `segmenter_module.py` указан ваш API-ключ и выбран провайдер.

In [ ]:
DATASET_PATH = 'merged_1209.json'
N_SAMPLES = 15

results = run_experiment(DATASET_PATH, n_samples=N_SAMPLES, seed=42)
summary = summarize_results(results)

print('\n=== Итоги ===')
for strat, s in summary.items():
    print(f"{strat:12s} | успех {s['success_rate']}% | спанов {s['avg_spans']} | ошибок {s['avg_errors']}")

In [ ]:
# Переводим результаты в DataFrame
df = pd.DataFrame(results)
print(f'Всего записей: {len(df)}')
print(f'Стратегии: {df["strategy"].unique()}')
df.head(3)

## 2. Успешность парсинга JSON по стратегиям

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

strategies = ['baseline', 'structured', 'few_shot']
colors = ['#e74c3c', '#2ecc71', '#3498db']
labels_ru = ['Baseline', 'Structured', 'Few-Shot']

# --- График 1: Success rate ---
success_rates = [summary[s]['success_rate'] for s in strategies]
bars = axes[0].bar(labels_ru, success_rates, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_ylim(0, 115)
axes[0].set_title('Успешность парсинга JSON', fontsize=13, fontweight='bold')
axes[0].set_ylabel('% успешных запросов')
for bar, val in zip(bars, success_rates):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'{val}%', ha='center', va='bottom', fontsize=12, fontweight='bold')

# --- График 2: Среднее количество спанов ---
avg_spans = [summary[s]['avg_spans'] for s in strategies]
bars2 = axes[1].bar(labels_ru, avg_spans, color=colors, edgecolor='white', linewidth=1.5)
axes[1].set_title('Среднее кол-во спанов', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Спанов на отзыв')
for bar, val in zip(bars2, avg_spans):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{val}', ha='center', va='bottom', fontsize=12, fontweight='bold')

# --- График 3: Среднее время ответа ---
for strat, color, label in zip(strategies, colors, labels_ru):
    times = df[df['strategy'] == strat]['elapsed_sec']
    axes[2].scatter([label] * len(times), times, color=color, alpha=0.6, s=60)
    axes[2].hlines(times.mean(), 
                   [label], [label],  # dummy, just for legend
                   linewidth=0)
    axes[2].axhline(y=times.mean(), 
                    xmin=strategies.index(strat)/3 + 0.05, 
                    xmax=(strategies.index(strat)+1)/3 - 0.05,
                    color=color, linewidth=2.5, linestyle='--')
axes[2].set_title('Время ответа API (сек)', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Секунды')

plt.tight_layout()
plt.savefig('results_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('График сохранён: results_overview.png')

## 3. Распределение количества спанов (boxplot)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

data_by_strategy = [
    df[df['strategy'] == s]['n_spans'].tolist()
    for s in strategies
]

bp = ax.boxplot(data_by_strategy, labels=labels_ru, patch_artist=True,
                medianprops=dict(color='black', linewidth=2))

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title('Распределение количества спанов по стратегиям', fontsize=14, fontweight='bold')
ax.set_ylabel('Количество спанов')
ax.set_xlabel('Стратегия промптинга')

plt.tight_layout()
plt.savefig('results_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Топ-10 меток по частоте (structured + few_shot)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, strat, color, label in zip(
    axes, ['structured', 'few_shot'], ['#2ecc71', '#3498db'], ['Structured', 'Few-Shot']
):
    label_counts = Counter()
    for row in df[df['strategy'] == strat].itertuples():
        for span in row.validation['spans']:
            label_counts[span['label']] += 1

    top10 = label_counts.most_common(10)
    if not top10:
        ax.set_title(f'{label}: нет данных')
        continue

    labels_list, counts = zip(*top10)
    bars = ax.barh(list(reversed(labels_list)), list(reversed(counts)),
                   color=color, alpha=0.8, edgecolor='white')
    ax.set_title(f'Топ-10 меток — {label}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Количество вхождений')
    for bar, val in zip(bars, list(reversed(counts))):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                str(val), va='center', fontsize=10)

plt.tight_layout()
plt.savefig('results_labels.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Распределение sentiment по стратегиям

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sentiment_colors = {'positive': '#2ecc71', 'negative': '#e74c3c', 'neutral': '#95a5a6'}

for ax, strat, label in zip(axes, ['structured', 'few_shot'], ['Structured', 'Few-Shot']):
    sent_counts = Counter()
    for row in df[df['strategy'] == strat].itertuples():
        for span in row.validation['spans']:
            sent_counts[span.get('sentiment', 'neutral')] += 1

    if not sent_counts:
        continue

    keys = list(sent_counts.keys())
    vals = list(sent_counts.values())
    clrs = [sentiment_colors.get(k, '#bdc3c7') for k in keys]
    wedges, texts, autotexts = ax.pie(
        vals, labels=keys, colors=clrs, autopct='%1.1f%%',
        startangle=90, textprops={'fontsize': 11}
    )
    for at in autotexts:
        at.set_fontweight('bold')
    ax.set_title(f'Sentiment — {label}', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('results_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Сводная таблица результатов

In [ ]:
rows = []
for strat in strategies:
    s = summary[strat]
    subset = df[df['strategy'] == strat]
    rows.append({
        'Стратегия': strat,
        'Успех парсинга': f"{s['success_rate']}%",
        'Спанов (avg)': s['avg_spans'],
        'Ошибок (avg)': s['avg_errors'],
        'Исправлено (avg)': s['avg_fixed'],
        'Время (avg, с)': round(subset['elapsed_sec'].mean(), 2),
        'Всего запросов': len(subset),
    })

summary_df = pd.DataFrame(rows).set_index('Стратегия')
summary_df

## 7. Выводы

**Baseline** полностью непригоден для машинной обработки: 0% успешного парсинга. Модель отвечает свободным текстом — без гарантий формата.

**Structured** и **Few-Shot** дают 100% корректный JSON. Guardrails не зафиксировали ни одной невалидной метки — whitelist работает.

**Few-Shot** в среднем выдаёт схожее количество спанов с Structured, но работает быстрее на коротких отзывах — примеры помогают модели сразу перейти к нужному формату без длинного описания схемы.

**Практический вывод:** для промышленного применения достаточно стратегии `structured` — она не требует поддержки примеров и даёт стабильный результат.